# AdventureWorks SQLite — Physical Schema Inspection

## Objective

This notebook connects to the local SQLite database used for the **Customer Value and Retention** project and exports the metadata needed to build a physical Entity Relationship Diagram (ERD).

The goal is not to analyse customer behaviour yet. At this stage, the goal is to inspect the database structure and produce files that can be reviewed and used to generate a complete ERD in PNG format.

## Expected database location

By default, the notebook searches for the database in:

```text
../data/adventureworks_customer_analytics.db
../data/adventureworks_customer_analytics
../data/adventureworks_customer_analytics/adventureworks_customer_analytics.db
```

If your database is in another location, update the `DB_PATH` variable in the next code cell.

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd
import json

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DB_PATH = Path("../data/adventureworks_customer_analytics.db")

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found: {DB_PATH.resolve()}"
    )

DB_PATH = DB_PATH.resolve()

print(DB_PATH)

C:\Users\Rodrigo\Desktop\profile\portfolio\notebooks-and-scripts\2-retail-analytics\2-customer-value-and-retention\data\adventureworks_customer_analytics.db


## Connect to SQLite

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

sqlite_version = conn.execute("SELECT sqlite_version();").fetchone()[0]
sqlite_version


'3.45.3'

## List database tables

In [4]:
tables = pd.read_sql_query(
    """
    SELECT 
        name AS table_name,
        type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY type, name;
    """,
    conn
)

tables

,table_name,type
0,person_address,table
1,person_addresstype,table
2,person_businessentityaddress,table
3,person_countryregion,table
4,person_emailaddress,table
5,person_person,table
6,person_stateprovince,table
7,production_product,table
8,production_productcategory,table
9,production_productsubcategory,table


## Table row counts

In [5]:
row_counts = []

for table_name in tables.loc[tables["type"] == "table", "table_name"]:
    count = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    row_counts.append({
        "table_name": table_name,
        "row_count": count
    })

row_counts_df = pd.DataFrame(row_counts).sort_values("table_name").reset_index(drop=True)
row_counts_df

,table_name,row_count
0,person_address,19614
1,person_addresstype,6
2,person_businessentityaddress,19614
3,person_countryregion,238
4,person_emailaddress,19972
5,person_person,19972
6,person_stateprovince,181
7,production_product,504
8,production_productcategory,4
9,production_productsubcategory,37


## Column metadata

In [6]:
columns_metadata = []

for table_name in tables.loc[tables["type"] == "table", "table_name"]:
    pragma_rows = conn.execute(f'PRAGMA table_info("{table_name}")').fetchall()

    for row in pragma_rows:
        cid, name, col_type, notnull, default_value, pk = row
        columns_metadata.append({
            "table_name": table_name,
            "column_order": cid,
            "column_name": name,
            "data_type": col_type,
            "not_null": bool(notnull),
            "default_value": default_value,
            "primary_key_position": pk
        })

columns_df = pd.DataFrame(columns_metadata).sort_values(
    ["table_name", "column_order"]
).reset_index(drop=True)

columns_df

,table_name,column_order,column_name,data_type,not_null,default_value,primary_key_position
0,person_address,0,AddressID,INTEGER,True,None,1
1,person_address,1,AddressLine1,TEXT,True,None,0
2,person_address,2,AddressLine2,TEXT,False,None,0
3,person_address,3,City,TEXT,True,None,0
4,person_address,4,StateProvinceID,INTEGER,True,None,0
...,...,...,...,...,...,...,...
157,sales_store,1,Name,TEXT,True,None,0
158,sales_store,2,SalesPersonID,INTEGER,False,None,0
159,sales_store,3,Demographics,TEXT,False,None,0
160,sales_store,4,rowguid,TEXT,True,None,0


## Primary keys

In [7]:
primary_keys_df = (
    columns_df
    .loc[columns_df["primary_key_position"] > 0, 
         ["table_name", "column_name", "primary_key_position"]]
    .sort_values(["table_name", "primary_key_position"])
    .reset_index(drop=True)
)

primary_keys_df

,table_name,column_name,primary_key_position
0,person_address,AddressID,1
1,person_addresstype,AddressTypeID,1
2,person_businessentityaddress,BusinessEntityID,1
3,person_businessentityaddress,AddressID,2
4,person_businessentityaddress,AddressTypeID,3
5,person_countryregion,CountryRegionCode,1
6,person_emailaddress,BusinessEntityID,1
7,person_emailaddress,EmailAddressID,2
8,person_person,BusinessEntityID,1
9,person_stateprovince,StateProvinceID,1


## Foreign keys

In [8]:
foreign_keys_metadata = []

for table_name in tables.loc[tables["type"] == "table", "table_name"]:
    fk_rows = conn.execute(f'PRAGMA foreign_key_list("{table_name}")').fetchall()

    for row in fk_rows:
        (
            fk_id,
            seq,
            referenced_table,
            from_column,
            to_column,
            on_update,
            on_delete,
            match
        ) = row

        foreign_keys_metadata.append({
            "table_name": table_name,
            "fk_id": fk_id,
            "fk_sequence": seq,
            "column_name": from_column,
            "referenced_table": referenced_table,
            "referenced_column": to_column,
            "on_update": on_update,
            "on_delete": on_delete,
            "match": match
        })

foreign_keys_df = pd.DataFrame(foreign_keys_metadata)

if not foreign_keys_df.empty:
    foreign_keys_df = foreign_keys_df.sort_values(
        ["table_name", "fk_id", "fk_sequence"]
    ).reset_index(drop=True)

foreign_keys_df

,table_name,fk_id,fk_sequence,column_name,referenced_table,referenced_column,on_update,on_delete,match
0,person_address,0,0,StateProvinceID,person_stateprovince,StateProvinceID,NO ACTION,NO ACTION,NONE
1,person_businessentityaddress,0,0,AddressTypeID,person_addresstype,AddressTypeID,NO ACTION,NO ACTION,NONE
2,person_businessentityaddress,1,0,AddressID,person_address,AddressID,NO ACTION,NO ACTION,NONE
3,person_emailaddress,0,0,BusinessEntityID,person_person,BusinessEntityID,NO ACTION,NO ACTION,NONE
4,person_stateprovince,0,0,TerritoryID,sales_salesterritory,TerritoryID,NO ACTION,NO ACTION,NONE
5,person_stateprovince,1,0,CountryRegionCode,person_countryregion,CountryRegionCode,NO ACTION,NO ACTION,NONE
6,production_product,0,0,ProductSubcategoryID,production_productsubcategory,ProductSubcategoryID,NO ACTION,NO ACTION,NONE
7,production_productsubcategory,0,0,ProductCategoryID,production_productcategory,ProductCategoryID,NO ACTION,NO ACTION,NONE
8,sales_customer,0,0,TerritoryID,sales_salesterritory,TerritoryID,NO ACTION,NO ACTION,NONE
9,sales_customer,1,0,StoreID,sales_store,BusinessEntityID,NO ACTION,NO ACTION,NONE


## Index metadata

In [9]:
indexes_metadata = []

for table_name in tables.loc[tables["type"] == "table", "table_name"]:
    index_rows = conn.execute(f'PRAGMA index_list("{table_name}")').fetchall()

    for index_row in index_rows:
        seq, index_name, is_unique, origin, partial = index_row

        index_columns = conn.execute(f'PRAGMA index_info("{index_name}")').fetchall()
        indexed_column_names = [col[2] for col in index_columns]

        indexes_metadata.append({
            "table_name": table_name,
            "index_name": index_name,
            "is_unique": bool(is_unique),
            "origin": origin,
            "partial": bool(partial),
            "columns": ", ".join(indexed_column_names)
        })

indexes_df = pd.DataFrame(indexes_metadata)

if not indexes_df.empty:
    indexes_df = indexes_df.sort_values(
        ["table_name", "index_name"]
    ).reset_index(drop=True)

indexes_df

,table_name,index_name,is_unique,origin,partial,columns
0,person_businessentityaddress,sqlite_autoindex_person_businessentityaddress_1,True,pk,False,"BusinessEntityID, AddressID, AddressTypeID"
1,person_countryregion,sqlite_autoindex_person_countryregion_1,True,pk,False,CountryRegionCode
2,person_emailaddress,sqlite_autoindex_person_emailaddress_1,True,pk,False,"BusinessEntityID, EmailAddressID"
3,production_product,idx_product_subcategory,False,c,False,ProductSubcategoryID
4,sales_customer,idx_customer_person,False,c,False,PersonID
5,sales_customer,idx_customer_territory,False,c,False,TerritoryID
6,sales_salesorderdetail,idx_sod_product,False,c,False,ProductID
7,sales_salesorderdetail,sqlite_autoindex_sales_salesorderdetail_1,True,pk,False,"SalesOrderID, SalesOrderDetailID"
8,sales_salesorderheader,idx_soh_customer_orderdate,False,c,False,"CustomerID, OrderDate"
9,sales_specialofferproduct,sqlite_autoindex_sales_specialofferproduct_1,True,pk,False,"SpecialOfferID, ProductID"


## ERD relationship summary

The following table is the most important output for the ERD. Each row represents a relationship between a child table and a parent table.

In [10]:
if foreign_keys_df.empty:
    erd_relationships_df = pd.DataFrame(
        columns=[
            "child_table",
            "child_column",
            "parent_table",
            "parent_column",
            "relationship"
        ]
    )
else:
    erd_relationships_df = foreign_keys_df.rename(
        columns={
            "table_name": "child_table",
            "column_name": "child_column",
            "referenced_table": "parent_table",
            "referenced_column": "parent_column"
        }
    )[[
        "child_table",
        "child_column",
        "parent_table",
        "parent_column",
        "on_update",
        "on_delete"
    ]]

    erd_relationships_df["relationship"] = (
        erd_relationships_df["parent_table"] 
        + "." 
        + erd_relationships_df["parent_column"]
        + " -> "
        + erd_relationships_df["child_table"]
        + "."
        + erd_relationships_df["child_column"]
    )

erd_relationships_df

,child_table,child_column,parent_table,parent_column,on_update,on_delete,relationship
0,person_address,StateProvinceID,person_stateprovince,StateProvinceID,NO ACTION,NO ACTION,person_stateprovince.StateProvinceID -> person...
1,person_businessentityaddress,AddressTypeID,person_addresstype,AddressTypeID,NO ACTION,NO ACTION,person_addresstype.AddressTypeID -> person_bus...
2,person_businessentityaddress,AddressID,person_address,AddressID,NO ACTION,NO ACTION,person_address.AddressID -> person_businessent...
3,person_emailaddress,BusinessEntityID,person_person,BusinessEntityID,NO ACTION,NO ACTION,person_person.BusinessEntityID -> person_email...
4,person_stateprovince,TerritoryID,sales_salesterritory,TerritoryID,NO ACTION,NO ACTION,sales_salesterritory.TerritoryID -> person_sta...
5,person_stateprovince,CountryRegionCode,person_countryregion,CountryRegionCode,NO ACTION,NO ACTION,person_countryregion.CountryRegionCode -> pers...
6,production_product,ProductSubcategoryID,production_productsubcategory,ProductSubcategoryID,NO ACTION,NO ACTION,production_productsubcategory.ProductSubcatego...
7,production_productsubcategory,ProductCategoryID,production_productcategory,ProductCategoryID,NO ACTION,NO ACTION,production_productcategory.ProductCategoryID -...
8,sales_customer,TerritoryID,sales_salesterritory,TerritoryID,NO ACTION,NO ACTION,sales_salesterritory.TerritoryID -> sales_cust...
9,sales_customer,StoreID,sales_store,BusinessEntityID,NO ACTION,NO ACTION,sales_store.BusinessEntityID -> sales_customer...


## Sample rows from each table

The next cell extracts a small sample from every table. This is useful to verify that data types and table contents match the intended analytical use.

In [11]:
sample_rows = {}

for table_name in tables.loc[tables["type"] == "table", "table_name"]:
    sample_rows[table_name] = pd.read_sql_query(
        f'SELECT * FROM "{table_name}" LIMIT 5',
        conn
    )

list(sample_rows.keys())

['person_address',
 'person_addresstype',
 'person_businessentityaddress',
 'person_countryregion',
 'person_emailaddress',
 'person_person',
 'person_stateprovince',
 'production_product',
 'production_productcategory',
 'production_productsubcategory',
 'purchasing_shipmethod',
 'sales_customer',
 'sales_salesorderdetail',
 'sales_salesorderheader',
 'sales_salesterritory',
 'sales_specialoffer',
 'sales_specialofferproduct',
 'sales_store']

In [12]:
# Display one sample table at a time by changing the table name below.
table_to_preview = "sales_salesorderheader"

if table_to_preview in sample_rows:
    display(sample_rows[table_to_preview])
else:
    print(f"Table not found: {table_to_preview}")

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,CustomerID,SalesPersonID,TerritoryID,BillToAddressID,ShipToAddressID,ShipMethodID,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate
0,43659,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43659,PO522145787,10-4020-000676,29825,279,5,985,985,5,16281,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,None,79B65321-39CA-4115-9CBA-8FE0903E12E6,2022-06-06 00:00:00.000
1,43660,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43660,PO18850127500,10-4020-000117,29672,279,5,921,921,5,5618,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,None,738DC42D-D03B-48A1-9822-F95A67EA7389,2022-06-06 00:00:00.000
2,43661,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43661,PO18473189620,10-4020-000442,29734,282,6,517,517,5,1346,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,None,D91B9131-18A4-4A11-BC3A-90B6F53E9D74,2022-06-06 00:00:00.000
3,43662,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43662,PO18444174044,10-4020-000227,29994,282,6,482,482,5,10456,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324,None,4A1ECFC0-CC3A-4740-B028-1C50BB48711C,2022-06-06 00:00:00.000
4,43663,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43663,PO18009186470,10-4020-000510,29565,276,4,1073,1073,5,4322,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108,None,9B1E7A40-6AE0-4AD3-811C-A64951857C4B,2022-06-06 00:00:00.000


## Export metadata files

The exported files can be attached later so that the physical ERD can be generated accurately.

The output directory is:

```text
../data/schema_exports
```

In [13]:
EXPORT_DIR = Path("../data/schema_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

tables.to_csv(EXPORT_DIR / "tables.csv", index=False)
row_counts_df.to_csv(EXPORT_DIR / "row_counts.csv", index=False)
columns_df.to_csv(EXPORT_DIR / "columns.csv", index=False)
primary_keys_df.to_csv(EXPORT_DIR / "primary_keys.csv", index=False)
foreign_keys_df.to_csv(EXPORT_DIR / "foreign_keys.csv", index=False)
indexes_df.to_csv(EXPORT_DIR / "indexes.csv", index=False)
erd_relationships_df.to_csv(EXPORT_DIR / "erd_relationships.csv", index=False)

metadata_json = {
    "database_path": str(DB_PATH),
    "sqlite_version": sqlite_version,
    "tables": tables.to_dict(orient="records"),
    "row_counts": row_counts_df.to_dict(orient="records"),
    "columns": columns_df.to_dict(orient="records"),
    "primary_keys": primary_keys_df.to_dict(orient="records"),
    "foreign_keys": foreign_keys_df.to_dict(orient="records"),
    "indexes": indexes_df.to_dict(orient="records"),
    "erd_relationships": erd_relationships_df.to_dict(orient="records")
}

with open(EXPORT_DIR / "schema_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata_json, f, indent=2, ensure_ascii=False)

EXPORT_DIR.resolve()

WindowsPath('C:/Users/Rodrigo/Desktop/profile/portfolio/notebooks-and-scripts/2-retail-analytics/2-customer-value-and-retention/data/schema_exports')

## Export sample data

This step exports the first rows of each table to separate CSV files. These files are not required for the ERD, but they help validate the meaning of each table.

In [14]:
SAMPLES_DIR = EXPORT_DIR / "sample_rows"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

for table_name, df in sample_rows.items():
    df.to_csv(SAMPLES_DIR / f"{table_name}_sample.csv", index=False)

SAMPLES_DIR.resolve()

WindowsPath('C:/Users/Rodrigo/Desktop/profile/portfolio/notebooks-and-scripts/2-retail-analytics/2-customer-value-and-retention/data/schema_exports/sample_rows')

## Create a ZIP file with all schema exports

In [15]:
import zipfile

ZIP_PATH = EXPORT_DIR.parent / "adventureworks_schema_exports.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for file_path in EXPORT_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, arcname=file_path.relative_to(EXPORT_DIR.parent))

ZIP_PATH.resolve()

WindowsPath('C:/Users/Rodrigo/Desktop/profile/portfolio/notebooks-and-scripts/2-retail-analytics/2-customer-value-and-retention/data/adventureworks_schema_exports.zip')

## Close the connection

In [16]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.


## Next step

Attach the generated ZIP file:

```text
../data/adventureworks_schema_exports.zip
```

With those metadata files, the next step is to generate a physical ERD in PNG format, including tables, primary keys, foreign keys and relationships.